In [1]:
import pandas as pd # Importamos librería pandas para crear el DataFrame
import json # Importamos librería JSON para leer archivo de origen de datos

# Importamos librería mysql.connector para conectarnos al servidor
import mysql.connector
from mysql.connector import errorcode

In [ ]:
# Fase 1.1. Obtención de los 5 genéros del archivo definitivo.json filtrados de 2020-2025

# Función para filtrar el archivo definitivo.json en función del género y rango de años
def creacion_fichero_genero(genre, año_inicio, año_fin):

    # Abrimos el JSON con todos los registros.
    with open("definitivo.json", "r", encoding="utf-8") as f:
        musica_json = json.load(f)

    # Creamos lista vacía para guardar los registros por género
    datos_genero = []

    # Recorremos cada registro ("artista/track") del JSON
    for artista in musica_json:

        # Creamos las variables género y año
        genero = artista["genre"]
        año = artista["year"]
        
        # Aplicamos filtros (género y año)
        if genero == genre and año_inicio <= año <= año_fin:
            
            # Si se cumple el filtro, añadimos en nuestra lista vacía la siguiente información
            datos_genero.append({
                "id": artista["id"],
                "artist_name": artista["artist_name"],
                "track_name": artista["track_name"],
                "genre": genero,
                "year": año
            })

    # Convertimos la lista de diccionarios en un DataFrame para facilitar la visualización de los datos y exportarlos
    data_music = pd.DataFrame(datos_genero)

    # Exportamos el DataFrame a JSON. orient="records" para estructurar el JSON y que sea fácil de leer, al igual que indent=4. force_ascii=False mantiene tíldes y caracteres especiales
    data_music.to_json(f"data_music_filtrado_{genre}.json", orient="records", indent=4, force_ascii=False)

In [ ]:
# Pasamos el género y rango de años a la función para obtener un fichero JSON por cada género

creacion_fichero_genero("rock", 2020, 2025)

creacion_fichero_genero("latin", 2020, 2025)

creacion_fichero_genero("flamenco", 2020, 2025)

creacion_fichero_genero("rap", 2020, 2025)

creacion_fichero_genero("indie", 2020, 2025)

In [4]:
# Fase 1.2. Enriquecemos nuestro fichero de la fase anterior con nueva información de la API de Last.fm

import json # Lectura y escritura en JSON
import requests # Para llamar a la API Last.fm
import time # Para añadir pausas y no saturar la API

# Función que recibirá la API key, el fichero JSON filtrado (fase 1.1) y género
def info_artista_api (api_key, fichero_genero, genre):
    url = "https://ws.audioscrobbler.com/2.0/" # Endpoint base de la API Last.fm

    # Abrimos el archivo JSON generado en la fase 1.1
    with open(fichero_genero, "r", encoding="utf-8") as f:
        data_music_api = json.load(f) # Cargamos su contenido en una lista de diccionarios

    # Creamos lista de artistas únicos para evitar llamadas innecesarias a la API
    artistas_unicos = list(set([ # Extraemos el nombre del artista
        item["artist_name"]
        for item in data_music_api 
        if item["artist_name"]])) # Solo incluimos si el nombre no está vacío o no es None

    # Diccionario donde guardaremos la información obtenida de la API para cada artista
    info_artistas = {}

    # Recorremos cada artista único
    for artista in artistas_unicos:

        # Definimos los parámetros de la API
        parametros = {
            "method": "artist.getinfo",   # Método para obtener la biografía
            "artist": artista,            # Nombre del artista
            "api_key": api_key,           # API key
            "format": "json"              # Respuesta en formato JSON
        }

        # Valores por defecto si no encuentra biografía, listeneres, playcount o artistas similares
        biografia = ""
        listeners = ""
        playcount = ""
        similares = ""

        # Llamada a la API. Usamos try/except para prevenir erorres que no dependan del código
        try:
            response = requests.get(url, params=parametros, timeout=10) # Si la petición tarda +10s, cancelamos con timeout
            data_lastfm = response.json() # Convertimos la respuesta a JSON
        except:
            # Si hay error, guardamos valores vacíos para ese artista
            info_artistas[artista] = {
                "biografia": biografia,
                "listeners": listeners,
                "playcount": playcount,
                "similares": similares
            }
            time.sleep(0.05) # Pausa para no saturar la API
            continue # Saltamos al siguiente artista

        # Comprobamos que 'artist' existe en la respuesta
        if "artist" in data_lastfm:
            
            # Creamos una variable más corta, más fácil de usar
            artista_data = data_lastfm["artist"]

            # Comprobamos que existan la clave 'bio' y la subclave 'summary'
            if "bio" in artista_data and "summary" in artista_data["bio"]:
                biografia = artista_data["bio"]["summary"]

            # Comprobamos que existan las estadísticas 
            if "stats" in artista_data:

                # Verificamos que exista 'listeners' dentro de 'stats'
                if "listeners" in artista_data["stats"]:
                    listeners = artista_data["stats"]["listeners"]

                # Verificamos que exista 'playcounts' dentro de 'stats'
                if "playcount" in artista_data["stats"]:
                    playcount = artista_data["stats"]["playcount"]

            # Comprobamos que existan los artistas similares en 'artist'
            if "similar" in artista_data and "artist" in artista_data["similar"]:
                
                nombres = [] # Creamos una lista vacía para almacenar los artistas similares
                
                # Extraemos cada artista similar que devuelva la API
                for a in artista_data["similar"]["artist"]:
                    nombres.append(a["name"])
                
                # Convertimos la lista en un único string separado por comas
                similares = ", ".join(nombres)

        """ Guardamos la información extraida en el diccionario principal
        La clave es el nombre del artista y el valor es otro diccionario con sus datos """
        info_artistas[artista] = {
            "biografia": biografia,
            "listeners": listeners,
            "playcount": playcount,
            "similares": similares
        }

        # Pausa para no saturar a la API
        time.sleep(0.05)

    """ Guardamos el nuevo JSON
    Con ensure_ascii = False nos aseguramos de que mantenga las tildes y ñ. Con encoding="utf-8" mantenemos tildes """
    with open(f"lastfm_{genre}.json", "w", encoding="utf-8") as f:
        json.dump(info_artistas, f, indent=4, ensure_ascii=False)

In [5]:
# Obtenemos archivo enriquecido de género 'flamenco' con la API key de Lourdes

api_key_flamenco = "d79b38ce7ccd273a5e94f02f8dd3a299"
info_artista_api(api_key_flamenco, "data_music_filtrado_flamenco.json", "flamenco")

In [6]:
# Obtenemos archivo enriquecido de género 'indie' con la API key de Ana

api_key_indie = "6786e294b805290158a20e4430160911" # Clave de Last.fm
info_artista_api(api_key_indie, "data_music_filtrado_indie.json", "indie")

In [7]:
# Obtenemos archivo enriquecido de género 'rock' con la API key de Lorena

api_key_rock = "484e0e00eb4ec9d3093d5fc446733ed0"
info_artista_api(api_key_rock, "data_music_filtrado_rock.json", "rock")

In [8]:
# Obtenemos archivo enriquecido de género 'rap' con la API key de Laura

api_key_rap = "c46b71050be388c1720fb9983feb9494"
info_artista_api(api_key_rap, "data_music_filtrado_rap.json", "rap")

In [9]:
# Obtenemos archivo enriquecido de género 'latin' con la API key de Valeria

api_key_latin = "9bb42ab8825458126cac1b7815e878e7"
info_artista_api(api_key_latin, "data_music_filtrado_latin.json", "latin")

In [3]:
# Fase 2

In [106]:
#nos conectamos al servidor 
cnx = mysql.connector.connect(user="root",
                              password ="(*0MySQL1*)",
                              host="127.0.0.1",
                              database= "music_stream",
                              charset='utf-8',
                              read_timeout=300,
                              write_timeout=300
    )

In [107]:
#el cursos hace que envie y reciba datos del servidor

cursor = cnx.cursor()

In [108]:
cnx.database #vemos si está conectada a la bbdd que queremos.

'music_stream'

Abrimos el json de lastfm para poder importar primero los datos de artistas ya que es FK de la tabla canciones


# como tenemos un diccionario de diccionarios para tratarlo con un dataframe hemos decidido utilizarlo con python.
with open("lastfm_flamenco.json", "r",encoding="utf-8") as fichero:  # para el formato/ caracteres -> encoding="utf-8" 
    flamenco_artistas = json.load(fichero)

artistas =[]  #defino variable vacia.

#hacemos un for para iterar el json, dicc de dicc, usamos el items para poder coger la key y los values
for nombre,x in flamenco_artistas.items():
    artistas.append((
        nombre,
        x["biografia"],
        x["listeners"],
        x["similares"],
        x["playcount"]
    ))
   

In [109]:
artistas = []

In [110]:
artistas

[]

In [111]:
artistas =[]
import base64

def artistas_lastfm (nombre_fichero):

    with open(nombre_fichero, "r",encoding="utf-8") as fichero:  
        lastfm_artistas = json.load(fichero)


    for nombre,x in lastfm_artistas.items():
        # Codificar el texto en base64
        #texto_base64 = base64.b64encode(x["similares"].encode("utf-8")).decode("utf-8")

        artistas.append([
            nombre,
            x["biografia"],
            x["listeners"],
            x["similares"], #texto_base64
            x["playcount"]]
        )
   

In [102]:
#llamo a la funcion tantas veces como ficheros tenga y los guardo en artistas.
artistas_lastfm("lastfm_flamenco.json")


In [ ]:
artistas_lastfm("lastfm_indie.json")


In [ ]:
artistas_lastfm("lastfm_latin.json")


In [112]:
artistas_lastfm("lastfm_rap.json")


In [ ]:
artistas_lastfm("lastfm_rock.json")

In [103]:
artistas

[['Manuel Cerpa',
  ' <a href="https://www.last.fm/music/Manuel+Cerpa">Read more on Last.fm</a>',
  '190',
  'Onur Gündüzer, Daniel Casares, rancapino chico, Manolo Sanlucar, Paul Latin',
  '652'],
 ['Pastora Soler',
  'Pilar Sánchez Luque (born September 27, 1978 in Coria del Río, Seville), better known by her stage name Pastora Soler, is a Spanish singer. She is also a songwriter and her compositions usually mix copla or flamenco with or pop or electronic music. Soler represented Spain in the Eurovision Song Contest 2012 in Baku, Azerbaijan with "Quédate Conmigo (Stay With Me)".\n\nA precocious chanteuse, she started singing coplas and flamenco songs at several events when she was a child. <a href="https://www.last.fm/music/Pastora+Soler">Read more on Last.fm</a>',
  '48647',
  'Vanesa Martín, Edurne, Malú, Ruth Lorenzo, Isabel Pantoja',
  '767487'],
 ['Juan Solano Orchestra',
  ' <a href="https://www.last.fm/music/Juan+Solano+Orchestra">Read more on Last.fm</a>',
  '6',
  '',
  '11'

In [113]:
artistas[339]

['Magass',
 ' <a href="https://www.last.fm/music/Magass">Read more on Last.fm</a>',
 '18',
 '',
 '308']

In [114]:
sql_artistas = """
INSERT INTO artistas (nombre, biografia,listeners,similares,playcount)
VALUES (%s, %s, %s, %s, %s)
"""

In [116]:
artistas

[['CVPELLV',
  ' <a href="https://www.last.fm/music/CVPELLV">Read more on Last.fm</a>',
  '20735',
  '4eu3, DMNDZ, Heroes X Villains, Hash Tag, BLVCK',
  '105872'],
 ['Waste',
  'There are at least 5 bands named "Waste".\n\nThe First is black metal band from Sweden formed in 2003 by drummer P.J and singer/guitarplayer A.k.A. Soon, D.R joined the band on bass. Two demos were released with this lineup, "Waste" (2003) and "Banish Life Forever" (2004). In the winter of 2004 Patrik joined in bass, D.R moved on to playing guitar and A.k.A concentrated on vocals. But soon, D.R dropped out and A.k.A picked up the guitar again. A demo was released in the spring of 2005 called "No Room For Happiness Here" <a href="https://www.last.fm/music/Waste">Read more on Last.fm</a>',
  '17067',
  'Saltwound, Dead/Awake, The Hate Project, Distinguisher, Traitors',
  '163358'],
 ['Sir Charles',
  'Sir Charles® name and logo are registered trademarks within the entertainment industry.\n\nThere are some others

In [ ]:
for a in artistas:
    try:
        cursor.execute(sql_artistas,a)
        print(a[1]) #1 sería el nombre

    except:
        #si hay un error, para, no sigas.
        cnx.rollback()
        print(f"Hay un error en la inserción de {a[1]}")

    else:
        cnx.commit()
        print("commit realizado")

    finally:
        print("proceso terminado")

Hay un error en la inserción de  <a href="https://www.last.fm/music/CVPELLV">Read more on Last.fm</a>
proceso terminado
Hay un error en la inserción de There are at least 5 bands named "Waste".

The First is black metal band from Sweden formed in 2003 by drummer P.J and singer/guitarplayer A.k.A. Soon, D.R joined the band on bass. Two demos were released with this lineup, "Waste" (2003) and "Banish Life Forever" (2004). In the winter of 2004 Patrik joined in bass, D.R moved on to playing guitar and A.k.A concentrated on vocals. But soon, D.R dropped out and A.k.A picked up the guitar again. A demo was released in the spring of 2005 called "No Room For Happiness Here" <a href="https://www.last.fm/music/Waste">Read more on Last.fm</a>
proceso terminado
Hay un error en la inserción de Sir Charles® name and logo are registered trademarks within the entertainment industry.

There are some others using the same name though they are not him, so do not be taken back if the noise they make diff

In [115]:
cursor.executemany(sql_artistas, artistas)

DatabaseError: 1205 (HY000): Lock wait timeout exceeded; try restarting transaction

In [ ]:
'''# Intentar insertar los datos
for valor in artistas:
    cursor.execute(sql_artistas, artistas)
    break

# Comitar los cambios
cnx.commit()
# Cerrar el cursor y la conexión
cursor.close()
cnx.close()'''

MySQLInterfaceError: Python type list cannot be converted

In [ ]:
'''#cursor.executemany(sql_artistas, artistas)

for i in range(0, len(artistas), 20):
    batch = artistas[i:i+20]
    cursor.executemany(sql_artistas, batch)
    cnx.commit()'''

DatabaseError: 1205 (HY000): Lock wait timeout exceeded; try restarting transaction

In [105]:
cursor.close()

True

ProgrammingError: Cursor is not connected

In [ ]:
# Abro el json con Pandas y almaceno la información en un DataFrame

df_flamenco_canciones = pd.read_json("data_music_filtrado_flamenco.json")

df_flamenco_canciones.head()

,id,artist_name,track_name,genre,year
0,5EY4OgQAzbbmLcfX71CYph,Pastora Soler,Demasiado amor,flamenco,2020
1,7aWPY1wzpVdMUwptggaDCG,Pastora Soler,Non credere,flamenco,2020
2,44rf2YH4E2bItZmoyRLXhZ,David DeMaría,Días de sol,flamenco,2020
3,245yCf5jTX1Jgkbr960i6h,Sandra Carrasco,Resistiré,flamenco,2020
4,6wsGAmBh5t9opcPFwCRamC,Las Niñas,Cuento de la buena pipa,flamenco,2020


In [ ]:
type(df_flamenco_canciones)

pandas.DataFrame

In [ ]:
flamenco = list(df_flamenco_canciones.itertuples(index=False, name= None))


In [ ]:
print(flamenco[0])

NameError: name 'flamenco' is not defined

In [ ]:
# Query de inserción

sql = """
INSERT INTO canciones (id,artist_name,track_name,genre,year)
VALUES (%s, %s, %s, %s, %s)
"""

In [ ]:
# Inserción de valores

cursor.executemany(sql, flamenco)

IntegrityError: 1452 (23000): Cannot add or update a child row: a foreign key constraint fails (`music_stream`.`canciones`, CONSTRAINT `fk_canciones_artistas` FOREIGN KEY (`artist_name`) REFERENCES `artistas` (`nombre`))

In [ ]:
# Confirmación de la inserción

cnx.commit()

In [ ]:
# Reviso los registros tras la inserción

cursor.execute("SELECT * FROM canciones;")
resultados = cursor.fetchall()